# Model matrix analysis

Compare multiple AI sentiment models on `output/processed/challenges.csv` +
`expectations.csv` to pick what works best for this dataset.

**Metrics**
- **prior_align** — challenges→negative, expectations→positive
- **gold_acc** — accuracy vs high-confidence keyword heuristic labels
- **neutral_rate** — lower is usually better for this use case
- **mean_margin / confidence** — decision certainty
- **sec_per_text** — speed
- **agreement / κ matrices** — how often models agree with each other

Composite rank weights gold + prior most heavily.


In [1]:
%pip install -q -r requirements.txt
%pip install -q "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

sys.path.insert(0, str(Path(".").resolve()))
from setup.model_evaluation import (
    MODEL_CANDIDATES,
    run_matrix,
    save_matrix_outputs,
)

OUT_DIR = Path("./output/model_matrix")
FIG_DIR = Path("./output/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Start with 60/side for a fast bake-off; set to None for full corpus
N_PER_SOURCE = 60

print("Candidates:")
for name, mid, kind in MODEL_CANDIDATES:
    print(f"  {name:20s}  {kind:12s}  {mid}")


Candidates:
  deberta-zs-v1.1       zero-shot     MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33
  deberta-mnli          zero-shot     MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  distilbert-mnli       zero-shot     typeform/distilbert-base-uncased-mnli
  twitter-roberta       sentiment3    cardiffnlp/twitter-roberta-base-sentiment-latest
  distilbert-7class     sentiment7    Thi144/sentiment-distilbert-7class
  sst2-distilbert       sentiment2    distilbert-base-uncased-finetuned-sst-2-english


## 1. Run the bake-off

First download can take a few minutes. Failed models are kept in the scorecard with an `error`.


In [3]:
bundle = run_matrix(n_per_source=N_PER_SOURCE)
out = save_matrix_outputs(bundle, OUT_DIR)

print(f"\nWinner: {bundle['winner']}")
print(f"Saved → {out}")
display(bundle["ranked"])


Evaluating 120 records (60 challenges, 60 expectations); heuristic gold labels: 29; isolated=True
Running deberta-zs-v1.1 (zero-shot) …
  done in 144.2s (1.202s/text) prior_align=0.325
Running deberta-mnli (zero-shot) …
  done in 137.6s (1.147s/text) prior_align=0.525
Running distilbert-mnli (zero-shot) …
  done in 5.0s (0.042s/text) prior_align=0.4917
Running twitter-roberta (sentiment3) …
  done in 3.8s (0.032s/text) prior_align=0.1333
Running distilbert-7class (sentiment7) …
  done in 1.4s (0.012s/text) prior_align=0.4667
Running sst2-distilbert (sentiment2) …
  done in 2.0s (0.017s/text) prior_align=0.5333

Winner: distilbert-7class
Saved → output/model_matrix


,model,model_id,kind,error,n,prior_align,gold_acc,gold_n,neutral_rate,mean_confidence,mean_margin,sec_per_text,total_sec,score,rank
0,distilbert-7class,Thi144/sentiment-distilbert-7class,sentiment7,None,120,0.4667,0.7586,29,0.0000,0.4217,0.4217,0.0116,1.39,0.723144,1
1,deberta-mnli,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,zero-shot,None,120,0.5250,0.7241,29,0.0917,0.6431,0.3736,1.1468,137.61,0.587123,2
2,sst2-distilbert,distilbert-base-uncased-finetuned-sst-2-english,sentiment2,None,120,0.5333,0.6207,29,0.0000,0.9378,0.9378,0.0167,2.01,0.571144,3
3,distilbert-mnli,typeform/distilbert-base-uncased-mnli,zero-shot,None,120,0.4917,0.6207,29,0.0250,0.6381,0.3876,0.0418,5.02,0.214552,4
4,deberta-zs-v1.1,MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33,zero-shot,None,120,0.3250,0.5517,29,0.4500,0.6281,0.3842,1.2020,144.23,-0.649760,5
5,twitter-roberta,cardiffnlp/twitter-roberta-base-sentiment-latest,sentiment3,None,120,0.1333,0.4138,29,0.7750,0.7595,0.7595,0.0317,3.81,-1.446203,6


## 2. Scorecard

In [4]:
cards = bundle["scorecard"].copy()
display(cards)

ok = cards.loc[cards["error"].isna()].copy()
metrics = ["prior_align", "gold_acc", "mean_margin", "neutral_rate", "sec_per_text"]
long = ok.melt(
    id_vars=["model"],
    value_vars=[m for m in metrics if m in ok.columns],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    long,
    x="model",
    y="value",
    color="model",
    facet_col="metric",
    facet_col_wrap=3,
    title="Model metrics on IPS discovery sample",
    labels={"value": "Score", "model": ""},
)
fig.update_layout(height=620, showlegend=False)
fig.update_xaxes(tickangle=-35)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()
fig.write_html(FIG_DIR / "model_matrix_metrics.html", include_plotlyjs="cdn")


,model,model_id,kind,error,n,prior_align,gold_acc,gold_n,neutral_rate,mean_confidence,mean_margin,sec_per_text,total_sec
0,deberta-zs-v1.1,MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33,zero-shot,None,120,0.3250,0.5517,29,0.4500,0.6281,0.3842,1.2020,144.23
1,deberta-mnli,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,zero-shot,None,120,0.5250,0.7241,29,0.0917,0.6431,0.3736,1.1468,137.61
2,distilbert-mnli,typeform/distilbert-base-uncased-mnli,zero-shot,None,120,0.4917,0.6207,29,0.0250,0.6381,0.3876,0.0418,5.02
3,twitter-roberta,cardiffnlp/twitter-roberta-base-sentiment-latest,sentiment3,None,120,0.1333,0.4138,29,0.7750,0.7595,0.7595,0.0317,3.81
4,distilbert-7class,Thi144/sentiment-distilbert-7class,sentiment7,None,120,0.4667,0.7586,29,0.0000,0.4217,0.4217,0.0116,1.39
5,sst2-distilbert,distilbert-base-uncased-finetuned-sst-2-english,sentiment2,None,120,0.5333,0.6207,29,0.0000,0.9378,0.9378,0.0167,2.01


## 3. Agreement matrices

Diagonal = 1.0. High off-diagonal agreement means models make the same call;
low agreement means the dataset is model-sensitive — trust the ranked scorecard.


In [5]:
def heat(mat: pd.DataFrame, title: str, fname: str):
    fig = px.imshow(
        mat,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="Tealgrn",
        title=title,
        labels=dict(color="Score"),
        zmin=0,
        zmax=1,
    )
    fig.update_layout(height=520, xaxis_title="", yaxis_title="")
    fig.show()
    fig.write_html(FIG_DIR / fname, include_plotlyjs="cdn")


heat(
    bundle["agreement"],
    "Pairwise label agreement (fraction same)",
    "model_agreement_matrix.html",
)
heat(
    bundle["kappa"],
    "Pairwise Cohen's κ",
    "model_kappa_matrix.html",
)


## 4. Composite rank (best for this dataset)

In [6]:
ranked = bundle["ranked"]
fig = px.bar(
    ranked,
    x="score",
    y="model",
    orientation="h",
    text=ranked["rank"].astype(str),
    color="score",
    color_continuous_scale="Blues",
    title="Composite rank — higher = better fit for IPS discovery notes",
    hover_data={
        "prior_align": True,
        "gold_acc": True,
        "neutral_rate": True,
        "sec_per_text": True,
        "score": ":.3f",
    },
)
fig.update_layout(
    height=420,
    yaxis=dict(autorange="reversed", title=""),
    xaxis_title="Composite score (z-weighted)",
    coloraxis_showscale=False,
)
fig.show()
fig.write_html(FIG_DIR / "model_composite_rank.html", include_plotlyjs="cdn")

winner = bundle["winner"]
print(f"Best model for this dataset: {winner}")
print("Wire it into setup/analyze_records.py as SENTIMENT_MODEL if different from current.")
display(ranked[["rank", "model", "model_id", "prior_align", "gold_acc", "neutral_rate", "sec_per_text", "score"]])


Best model for this dataset: distilbert-7class
Wire it into setup/analyze_records.py as SENTIMENT_MODEL if different from current.


,rank,model,model_id,prior_align,gold_acc,neutral_rate,sec_per_text,score
0,1,distilbert-7class,Thi144/sentiment-distilbert-7class,0.4667,0.7586,0.0000,0.0116,0.723144
1,2,deberta-mnli,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,0.5250,0.7241,0.0917,1.1468,0.587123
2,3,sst2-distilbert,distilbert-base-uncased-finetuned-sst-2-english,0.5333,0.6207,0.0000,0.0167,0.571144
3,4,distilbert-mnli,typeform/distilbert-base-uncased-mnli,0.4917,0.6207,0.0250,0.0418,0.214552
4,5,deberta-zs-v1.1,MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33,0.3250,0.5517,0.4500,1.2020,-0.649760
5,6,twitter-roberta,cardiffnlp/twitter-roberta-base-sentiment-latest,0.1333,0.4138,0.7750,0.0317,-1.446203


## 5. Where models disagree

Inspect rows where the top-2 ranked models disagree — useful for picking a
tie-breaker or keyword override.


In [7]:
preds = bundle["predictions"]
ranked_ok = bundle["ranked"]

if len(ranked_ok) >= 2:
    a, b = ranked_ok.iloc[0]["model"], ranked_ok.iloc[1]["model"]
    disagree = preds.loc[preds[a] != preds[b], ["text", "source", "prior", "gold", a, b]].copy()
    print(f"{a} vs {b}: {len(disagree)} / {len(preds)} disagreements")
    display(disagree.head(20))
else:
    print("Need ≥2 successful models to compare disagreements.")


distilbert-7class vs deberta-mnli: 39 / 120 disagreements


,text,source,prior,gold,distilbert-7class,deberta-mnli
0,Workaround: Type into county GIS map or Camino,challenges,negative,negative,positive,negative
1,"Codes is doing service, mailing",challenges,negative,NaN,positive,neutral
2,Units! Subparcels,challenges,negative,NaN,positive,negative
9,Not much existing documentation for IPS – figu...,challenges,negative,negative,negative,neutral
13,Bill: Let’s not add more software – don't,challenges,negative,NaN,negative,positive
14,When you schedule an inspection on camino it d...,challenges,negative,NaN,negative,neutral
16,That’s one where violation is always the same ...,challenges,negative,NaN,positive,negative
20,"Camino – once passed, if missed that day",challenges,negative,NaN,negative,positive
21,Disconnected – permit review touches other dep...,challenges,negative,NaN,positive,negative
23,IPS auto-fills contact information on forms,challenges,negative,NaN,positive,neutral


## 6. Optional: full-corpus re-run of the winner

Uncomment to re-score the full dataset with only the winning model (slower).


In [8]:
# from setup.model_matrix import MODEL_CANDIDATES, run_matrix, save_matrix_outputs
#
# winner_name = bundle["winner"]
# winner_spec = [m for m in MODEL_CANDIDATES if m[0] == winner_name]
# full = run_matrix(n_per_source=None, models=winner_spec)
# save_matrix_outputs(full, OUT_DIR / "full_winner")
# display(full["scorecard"])
